<a href="https://colab.research.google.com/github/SANGHATI23/ohdsi-fhir-omop-showcase-demo/blob/main/FHIRy_pyOMOP_TFL_MedicationReferenceHardening_v14_FIXED.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FHIRy–pyOMOP TFL Submission Hardening

## P0.5 Medication attribution + P0.6 unresolved-reference micro-test

This notebook strengthens two prespecified TFL warning families without changing the V0–V5 study design.

- **P0.5:** deterministic medication attribution based on patient/reference/status/date evidence in the frozen source resources.
- **P0.6:** a focused unresolved-reference fault-injection micro-test (`REF_MICRO_01`) outside the V0–V5 numbering.

The notebook does not add an AI model, probabilistic mapping, ontology learning, or a new composite score.

# Phase A

## Load frozen v12 source packages

In [1]:
from pathlib import Path
from collections import defaultdict
import copy
import hashlib
import json
import re
import tarfile
import uuid

import numpy as np
import pandas as pd

from google.colab import drive
drive.mount("/content/drive")

MYDRIVE = Path("/content/drive/MyDrive")
RUN_ROOT = MYDRIVE / "fhir_omop_colab" / "tfl_execution_v6"
FREEZE_ROOT = RUN_ROOT / "submission_freeze_v12"
SOURCE_DIR = FREEZE_ROOT / "sources"
SCHEMA_DIR = FREEZE_ROOT / "schema"
OUTPUT_DIR = FREEZE_ROOT / "medication_reference_v14"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

VARIANTS = ["V0", "V1", "V2", "V3", "V4", "V5"]

ARCHIVES = {
    variant: SOURCE_DIR / f"{variant}_frozen_clinical_core_25k.tar.gz"
    for variant in VARIANTS
}

for variant, path in ARCHIVES.items():
    if not path.exists():
        raise FileNotFoundError(
            f"{path} is missing. Run the v12 freeze notebook first."
        )

warning_vocab_path = SCHEMA_DIR / "tfl_warning_vocabulary_v1.0.0.csv"

if not warning_vocab_path.exists():
    raise FileNotFoundError(
        f"{warning_vocab_path} is missing. Run v12 first."
    )

WARNING_VOCAB = pd.read_csv(warning_vocab_path)
WARNING_CODES = set(
    WARNING_VOCAB["warning_code"].astype(str)
)

required_warning_codes = {
    "W_MEDICATION_ATTRIBUTION",
    "W_UNRESOLVED_REFERENCE",
}

missing_warning_codes = (
    required_warning_codes
    - WARNING_CODES
)

if missing_warning_codes:
    raise RuntimeError(
        "Frozen warning vocabulary is missing: "
        + ", ".join(sorted(missing_warning_codes))
    )

print("Freeze root:", FREEZE_ROOT)
print("Output dir:", OUTPUT_DIR)

Mounted at /content/drive
Freeze root: /content/drive/MyDrive/fhir_omop_colab/tfl_execution_v6/submission_freeze_v12
Output dir: /content/drive/MyDrive/fhir_omop_colab/tfl_execution_v6/submission_freeze_v12/medication_reference_v14


In [2]:
def read_archive_grouped(archive_path):
    grouped = defaultdict(list)

    with tarfile.open(archive_path, "r:gz") as tar:
        members = [
            member
            for member in tar.getmembers()
            if member.isfile()
            and member.name.lower().endswith(".ndjson")
        ]

        for member in members:
            handle = tar.extractfile(member)

            if handle is None:
                continue

            for raw_line in handle:
                line = raw_line.decode("utf-8").strip()

                if not line:
                    continue

                resource = json.loads(line)

                if resource.get("resourceType") == "Bundle":
                    for entry in resource.get("entry", []):
                        inner = entry.get("resource", {})
                        rt = inner.get("resourceType")

                        if rt:
                            grouped[str(rt)].append(inner)

                else:
                    rt = resource.get("resourceType")

                    if rt:
                        grouped[str(rt)].append(resource)

    return dict(grouped)


RAW = {
    variant: read_archive_grouped(path)
    for variant, path in ARCHIVES.items()
}


inventory = pd.DataFrame([
    {
        "variant": variant,
        "resource_type": resource_type,
        "rows": len(rows),
    }
    for variant, grouped in RAW.items()
    for resource_type, rows in grouped.items()
])

display(
    inventory[
        inventory["resource_type"]
        .astype(str)
        .str.startswith("Medication")
    ]
    .pivot_table(
        index="resource_type",
        columns="variant",
        values="rows",
        aggfunc="sum",
        fill_value=0,
    )
)

variant,V0,V1,V2,V3,V4,V5
resource_type,,,,,,
Medication,14131,14131,14131,14131,12718,14131
MedicationAdministration,14131,14131,14131,14131,12718,14131
MedicationRequest,25000,25000,25000,25000,25000,25000


# Phase B

## Deterministic medication-attribution rules

In [3]:

# Resource types that can represent patient-specific medication exposure/use.
PATIENT_MEDICATION_RESOURCE_TYPES = {
    "MedicationRequest",
    "MedicationAdministration",
    "MedicationStatement",
    "MedicationDispense",
    "MedicationUsage",
}

NON_PATIENT_MEDICATION_RESOURCE_TYPES = {
    "Medication",
}

EXCLUDED_STATUSES = {
    "entered-in-error",
}

DATE_PATHS = {
    "MedicationRequest": [
        ("authoredOn",),
        ("dispenseRequest", "validityPeriod", "start"),
        ("dispenseRequest", "validityPeriod", "end"),
    ],
    "MedicationAdministration": [
        ("effectiveDateTime",),
        ("effectivePeriod", "start"),
        ("effectivePeriod", "end"),
        ("occurenceDateTime",),
        ("occurencePeriod", "start"),
        ("occurencePeriod", "end"),
        ("occurrenceDateTime",),
        ("occurrencePeriod", "start"),
        ("occurrencePeriod", "end"),
    ],
    "MedicationStatement": [
        ("effectiveDateTime",),
        ("effectivePeriod", "start"),
        ("effectivePeriod", "end"),
        ("dateAsserted",),
    ],
    "MedicationDispense": [
        ("whenHandedOver",),
        ("whenPrepared",),
    ],
    "MedicationUsage": [
        ("dateAsserted",),
        ("effectiveDateTime",),
        ("effectivePeriod", "start"),
        ("effectivePeriod", "end"),
    ],
}

PATIENT_REFERENCE_PATHS = [
    ("subject", "reference"),
    ("patient", "reference"),
]

MEDICATION_REFERENCE_PATHS = [
    ("medicationReference", "reference"),
    ("medication", "reference"),
]

MEDICATION_CODE_PATHS = [
    ("medicationCodeableConcept", "coding"),
    ("medication", "concept", "coding"),
    ("medication", "coding"),
]


def get_path(obj, path):
    current = obj

    for key in path:
        if not isinstance(current, dict):
            return None

        current = current.get(key)

        if current is None:
            return None

    return current


def first_nonempty_path(obj, paths):
    for path in paths:
        value = get_path(obj, path)

        if value is None:
            continue

        if isinstance(value, str):
            if value.strip():
                return ".".join(path), value.strip()

        elif isinstance(value, (list, dict)):
            if len(value):
                return ".".join(path), value

        else:
            return ".".join(path), value

    return None, None


def parse_fhir_reference(reference):
    if not isinstance(reference, str):
        return None, None

    ref = reference.strip()

    if not ref:
        return None, None

    if ref.startswith("#"):
        return "#", ref[1:]

    ref = re.sub(r"/_history/[^/]+$", "", ref)

    parts = [
        part
        for part in ref.rstrip("/").split("/")
        if part
    ]

    if len(parts) < 2:
        return None, None

    return parts[-2], parts[-1]


def resource_index(grouped):
    return {
        (str(resource_type), str(resource["id"]))
        for resource_type, rows in grouped.items()
        for resource in rows
        if resource.get("id") is not None
    }


def patient_id_set(grouped):
    return {
        str(resource["id"])
        for resource in grouped.get("Patient", [])
        if resource.get("id") is not None
    }


def has_medication_code(resource):
    _, value = first_nonempty_path(
        resource,
        MEDICATION_CODE_PATHS,
    )

    return value is not None


# IMPORTANT PERFORMANCE FIX:
# Build these once per variant instead of once per medication row.
VARIANT_RESOURCE_INDEX = {
    variant: resource_index(RAW[variant])
    for variant in VARIANTS
}

VARIANT_PATIENT_IDS = {
    variant: patient_id_set(RAW[variant])
    for variant in VARIANTS
}

print("Precomputed source indexes:")
for variant in VARIANTS:
    print(
        variant,
        "resources =", len(VARIANT_RESOURCE_INDEX[variant]),
        "| patients =", len(VARIANT_PATIENT_IDS[variant]),
    )


def medication_attribution_record(
    resource,
    variant,
    source_index,
    patients,
):
    resource_type = str(
        resource.get("resourceType", "")
    )

    resource_id = (
        str(resource.get("id"))
        if resource.get("id") is not None
        else None
    )

    status = str(
        resource.get("status", "")
    ).strip().lower()

    patient_path, patient_reference = first_nonempty_path(
        resource,
        PATIENT_REFERENCE_PATHS,
    )

    patient_ref_type, patient_ref_id = parse_fhir_reference(
        patient_reference
    )

    patient_reference_present = (
        patient_reference is not None
    )

    patient_reference_is_patient = (
        patient_ref_type == "Patient"
    )

    patient_resolved = (
        patient_reference_is_patient
        and patient_ref_id in patients
    )

    medication_ref_path, medication_reference = first_nonempty_path(
        resource,
        MEDICATION_REFERENCE_PATHS,
    )

    medication_ref_type, medication_ref_id = parse_fhir_reference(
        medication_reference
    )

    medication_reference_present = (
        medication_reference is not None
    )

    medication_reference_resolved = (
        medication_reference_present
        and (
            medication_ref_type,
            medication_ref_id,
        ) in source_index
    )

    medication_code_present = has_medication_code(
        resource
    )

    date_path, date_value = first_nonempty_path(
        resource,
        DATE_PATHS.get(
            resource_type,
            [],
        ),
    )

    date_present = (
        date_value is not None
    )

    status_present = (
        bool(status)
    )

    status_excluded = (
        status in EXCLUDED_STATUSES
    )

    reasons = []

    if not patient_reference_present:
        reasons.append(
            "MISSING_PATIENT_REFERENCE"
        )

    elif not patient_reference_is_patient:
        reasons.append(
            "PATIENT_REFERENCE_WRONG_TYPE"
        )

    elif not patient_resolved:
        reasons.append(
            "UNRESOLVED_PATIENT_REFERENCE"
        )

    if (
        not medication_reference_present
        and not medication_code_present
    ):
        reasons.append(
            "MISSING_MEDICATION_EVIDENCE"
        )

    if (
        medication_reference_present
        and not medication_reference_resolved
    ):
        reasons.append(
            "UNRESOLVED_MEDICATION_REFERENCE"
        )

    if not status_present:
        reasons.append(
            "MISSING_STATUS"
        )

    if not date_present:
        reasons.append(
            "MISSING_DATE"
        )

    if status_excluded:
        reasons.append(
            "EXCLUDED_STATUS"
        )

    warning = (
        len(reasons) > 0
    )

    eligible_for_patient_mapping = (
        patient_resolved
        and (
            medication_code_present
            or medication_reference_resolved
        )
        and status_present
        and not status_excluded
        and date_present
    )

    return {
        "variant": variant,
        "source_resource_type": resource_type,
        "source_resource_id": resource_id,
        "patient_reference_path": patient_path,
        "patient_reference": patient_reference,
        "patient_reference_resolved": patient_resolved,
        "medication_reference_path": medication_ref_path,
        "medication_reference": medication_reference,
        "medication_reference_resolved": (
            medication_reference_resolved
        ),
        "medication_code_present": medication_code_present,
        "status": status or None,
        "date_path": date_path,
        "date_present": date_present,
        "eligible_for_patient_mapping": (
            eligible_for_patient_mapping
        ),
        "warning_code": (
            "W_MEDICATION_ATTRIBUTION"
            if warning
            else ""
        ),
        "warning_reasons": "|".join(
            sorted(set(reasons))
        ),
    }


Precomputed source indexes:
V0 resources = 154333 | patients = 1071
V1 resources = 154333 | patients = 1071
V2 resources = 153123 | patients = 1071
V3 resources = 154333 | patients = 1071
V4 resources = 151507 | patients = 1071
V5 resources = 154333 | patients = 1071


## Run attribution rules on frozen V0–V5

In [4]:

MEDICATION_ATTRIBUTION = {}

summary_rows = []

for variant in VARIANTS:
    rows = []

    source_index = VARIANT_RESOURCE_INDEX[variant]
    patients = VARIANT_PATIENT_IDS[variant]

    medication_resources = []

    for resource_type in sorted(
        PATIENT_MEDICATION_RESOURCE_TYPES
    ):
        medication_resources.extend(
            RAW[variant].get(
                resource_type,
                [],
            )
        )

    print(
        variant,
        "| medication resources:",
        f"{len(medication_resources):,}",
    )

    for i, resource in enumerate(
        medication_resources,
        start=1,
    ):
        rows.append(
            medication_attribution_record(
                resource=resource,
                variant=variant,
                source_index=source_index,
                patients=patients,
            )
        )

        if i % 10000 == 0:
            print(
                "  processed",
                f"{i:,}",
                "/",
                f"{len(medication_resources):,}",
            )

    df = pd.DataFrame(rows)

    if df.empty:
        raise RuntimeError(
            f"{variant}: no supported patient-specific medication resources found."
        )

    MEDICATION_ATTRIBUTION[variant] = df

    warning_count = int(
        (
            df["warning_code"]
            == "W_MEDICATION_ATTRIBUTION"
        ).sum()
    )

    summary_rows.append({
        "variant": variant,
        "medication_event_resources": len(df),
        "attribution_warning_rows": warning_count,
        "attribution_warning_rate": (
            warning_count / len(df)
        ),
        "eligible_for_patient_mapping": int(
            df[
                "eligible_for_patient_mapping"
            ].sum()
        ),
    })


MEDICATION_SUMMARY = pd.DataFrame(
    summary_rows
)

display(MEDICATION_SUMMARY)


V0 | medication resources: 39,131
  processed 10,000 / 39,131
  processed 20,000 / 39,131
  processed 30,000 / 39,131
V1 | medication resources: 39,131
  processed 10,000 / 39,131
  processed 20,000 / 39,131
  processed 30,000 / 39,131
V2 | medication resources: 39,131
  processed 10,000 / 39,131
  processed 20,000 / 39,131
  processed 30,000 / 39,131
V3 | medication resources: 39,131
  processed 10,000 / 39,131
  processed 20,000 / 39,131
  processed 30,000 / 39,131
V4 | medication resources: 37,718
  processed 10,000 / 37,718
  processed 20,000 / 37,718
  processed 30,000 / 37,718
V5 | medication resources: 39,131
  processed 10,000 / 39,131
  processed 20,000 / 39,131
  processed 30,000 / 39,131


,variant,medication_event_resources,attribution_warning_rows,attribution_warning_rate,eligible_for_patient_mapping
0,V0,39131,0,0.000000,39131
1,V1,39131,0,0.000000,39131
2,V2,39131,0,0.000000,39131
3,V3,39131,0,0.000000,39131
4,V4,37718,1003,0.026592,36715
5,V5,39131,0,0.000000,39131


In [5]:
# Controlled-method validation:
# V1, V2 and V3 do not alter medication domains and should
# reproduce the same medication-attribution profile as V0.
v0_row = (
    MEDICATION_SUMMARY[
        MEDICATION_SUMMARY["variant"]
        == "V0"
    ]
    .iloc[0]
)

v0_warning_count = int(
    v0_row["attribution_warning_rows"]
)

v0_warning_rate = float(
    v0_row["attribution_warning_rate"]
)

for variant in ["V1", "V2", "V3", "V5"]:
    row = (
        MEDICATION_SUMMARY[
            MEDICATION_SUMMARY["variant"]
            == variant
        ]
        .iloc[0]
    )

    # V5 combines Patient + Condition perturbations but does not
    # alter medication resources; attribution logic may only change
    # if its Patient perturbation breaks references, which this test
    # makes visible rather than silently accepting.
    if (
        variant in ["V1", "V2", "V3"]
        and (
            int(row["attribution_warning_rows"])
            != v0_warning_count
            or not np.isclose(
                float(row["attribution_warning_rate"]),
                v0_warning_rate,
            )
        )
    ):
        raise RuntimeError(
            f"{variant}: medication-attribution response changed "
            "despite no medication-domain perturbation."
        )


v4_row = (
    MEDICATION_SUMMARY[
        MEDICATION_SUMMARY["variant"]
        == "V4"
    ]
    .iloc[0]
)

if not (
    float(v4_row["attribution_warning_rate"])
    >
    v0_warning_rate
):
    raise RuntimeError(
        "V4 medication-attribution warning rate does not increase "
        "above frozen V0 under the hardened deterministic rule."
    )

print(
    "PASS: V1/V2/V3 medication attribution matches V0."
)

print(
    "PASS: V4 medication-attribution warning rate increases above V0."
)

PASS: V1/V2/V3 medication attribution matches V0.
PASS: V4 medication-attribution warning rate increases above V0.


## Warning-reason profile

In [6]:
reason_rows = []

for variant, df in MEDICATION_ATTRIBUTION.items():
    for value in df["warning_reasons"]:
        if not value:
            continue

        for reason in value.split("|"):
            reason_rows.append({
                "variant": variant,
                "reason": reason,
            })

MEDICATION_REASON_COUNTS = (
    pd.DataFrame(reason_rows)
    .groupby(
        ["variant", "reason"]
    )
    .size()
    .rename("rows")
    .reset_index()
)

display(
    MEDICATION_REASON_COUNTS
    .pivot_table(
        index="reason",
        columns="variant",
        values="rows",
        fill_value=0,
    )
)

variant,V4
reason,
UNRESOLVED_MEDICATION_REFERENCE,1003.0


# Phase C

## P0.6 unresolved-reference micro-test

In [7]:
def recursive_reference_paths(
    obj,
    prefix=(),
):
    found = []

    if isinstance(obj, dict):
        for key, value in obj.items():
            path = prefix + (str(key),)

            if (
                key == "reference"
                and isinstance(value, str)
                and value.strip()
            ):
                found.append(
                    (
                        path,
                        value.strip(),
                    )
                )

            found.extend(
                recursive_reference_paths(
                    value,
                    path,
                )
            )

    elif isinstance(obj, list):
        for index, value in enumerate(obj):
            found.extend(
                recursive_reference_paths(
                    value,
                    prefix + (str(index),),
                )
            )

    return found


def set_path_value(
    obj,
    path,
    value,
):
    current = obj

    for part in path[:-1]:
        if isinstance(current, list):
            current = current[int(part)]
        else:
            current = current[part]

    if isinstance(current, list):
        current[int(path[-1])] = value
    else:
        current[path[-1]] = value


V0_INDEX = resource_index(
    RAW["V0"]
)

preferred_resource_types = [
    "Encounter",
    "MedicationRequest",
    "Condition",
    "Observation",
    "Procedure",
]

candidate = None

for resource_type in preferred_resource_types:
    for resource in RAW["V0"].get(
        resource_type,
        [],
    ):
        for path, reference in (
            recursive_reference_paths(
                resource
            )
        ):
            ref_type, ref_id = (
                parse_fhir_reference(
                    reference
                )
            )

            if (
                ref_type == "Patient"
                and (
                    ref_type,
                    ref_id,
                ) in V0_INDEX
            ):
                candidate = {
                    "resource_type": resource_type,
                    "resource": resource,
                    "path": path,
                    "reference": reference,
                    "resolved_type": ref_type,
                    "resolved_id": ref_id,
                }
                break

        if candidate is not None:
            break

    if candidate is not None:
        break


if candidate is None:
    raise RuntimeError(
        "Could not find a resolvable Patient reference for the micro-test."
    )


baseline_resource = copy.deepcopy(
    candidate["resource"]
)

fault_resource = copy.deepcopy(
    candidate["resource"]
)

BROKEN_REFERENCE = (
    "Patient/TFL-UNRESOLVED-REFERENCE"
)

set_path_value(
    fault_resource,
    candidate["path"],
    BROKEN_REFERENCE,
)


print(
    "Selected resource:",
    candidate["resource_type"],
    candidate["resource"].get("id"),
)

print(
    "Reference path:",
    ".".join(candidate["path"]),
)

print(
    "Baseline reference:",
    candidate["reference"],
)

print(
    "Injected reference:",
    BROKEN_REFERENCE,
)

Selected resource: Encounter d98dbfa6-8476-1fee-6207-9f5da538e32b
Reference path: subject.reference
Baseline reference: Patient/d98dbfa6-8476-1fee-e0bf-1289cc0108ef
Injected reference: Patient/TFL-UNRESOLVED-REFERENCE


In [8]:
def reference_resolves(
    reference,
    source_index,
):
    ref_type, ref_id = (
        parse_fhir_reference(
            reference
        )
    )

    return (
        ref_type,
        ref_id,
    ) in source_index


baseline_reference = (
    candidate["reference"]
)

fault_reference = (
    BROKEN_REFERENCE
)

baseline_resolves = (
    reference_resolves(
        baseline_reference,
        V0_INDEX,
    )
)

fault_resolves = (
    reference_resolves(
        fault_reference,
        V0_INDEX,
    )
)

if not baseline_resolves:
    raise RuntimeError(
        "Baseline reference unexpectedly does not resolve."
    )

if fault_resolves:
    raise RuntimeError(
        "Injected broken reference unexpectedly resolves."
    )


MICRO_TEST_ID = "REF_MICRO_01"

fault_source_id = str(
    fault_resource.get("id")
)

transformation_name = (
    f"{MICRO_TEST_ID}|"
    f"{candidate['resource_type']}|"
    f"{fault_source_id}|"
    f"{'.'.join(candidate['path'])}|"
    f"{BROKEN_REFERENCE}|"
    "tfl-schema-1.0.0"
)

REF_MICRO_AUDIT = pd.DataFrame([
    {
        "transformation_id": str(
            uuid.uuid5(
                uuid.NAMESPACE_URL,
                transformation_name,
            )
        ),
        "variant": MICRO_TEST_ID,
        "source_resource_type": (
            candidate["resource_type"]
        ),
        "source_resource_id": (
            fault_source_id
        ),
        "source_reference_or_path": (
            ".".join(
                candidate["path"]
            )
        ),
        "source_value_or_code": (
            BROKEN_REFERENCE
        ),
        "target_omop_table": None,
        "target_omop_record_id": None,
        "target_omop_field": None,
        "mapping_rule_id": (
            "required-reference-resolution-gate"
        ),
        "fidelity_status": (
            "TRACEABILITY_LOSS"
        ),
        "warning_code": (
            "W_UNRESOLVED_REFERENCE"
        ),
        "transformation_version": (
            "TFL-1.0.0|reference-microtest-v14"
        ),
        "baseline_reference_resolved": (
            baseline_resolves
        ),
        "fault_reference_resolved": (
            fault_resolves
        ),
        "mapping_eligible": False,
    }
])

display(
    REF_MICRO_AUDIT
)


if (
    REF_MICRO_AUDIT.iloc[0][
        "warning_code"
    ]
    !=
    "W_UNRESOLVED_REFERENCE"
):
    raise RuntimeError(
        "Unresolved-reference warning was not emitted."
    )

if (
    REF_MICRO_AUDIT.iloc[0][
        "fidelity_status"
    ]
    !=
    "TRACEABILITY_LOSS"
):
    raise RuntimeError(
        "Unresolved required reference did not produce "
        "the expected lineage consequence."
    )

if pd.notna(
    REF_MICRO_AUDIT.iloc[0][
        "target_omop_record_id"
    ]
):
    raise RuntimeError(
        "Fault-injected record unexpectedly has a target link."
    )

print(
    "PASS: baseline reference resolves."
)

print(
    "PASS: injected reference is unresolved."
)

print(
    "PASS: W_UNRESOLVED_REFERENCE emitted."
)

print(
    "PASS: unresolved required reference is classified "
    "TRACEABILITY_LOSS and is not mapping-eligible."
)

,transformation_id,variant,source_resource_type,source_resource_id,source_reference_or_path,source_value_or_code,target_omop_table,target_omop_record_id,target_omop_field,mapping_rule_id,fidelity_status,warning_code,transformation_version,baseline_reference_resolved,fault_reference_resolved,mapping_eligible
0,e2a5d2eb-8e08-5f8b-a971-22399a505f4c,REF_MICRO_01,Encounter,d98dbfa6-8476-1fee-6207-9f5da538e32b,subject.reference,Patient/TFL-UNRESOLVED-REFERENCE,None,None,None,required-reference-resolution-gate,TRACEABILITY_LOSS,W_UNRESOLVED_REFERENCE,TFL-1.0.0|reference-microtest-v14,True,False,False


PASS: baseline reference resolves.
PASS: injected reference is unresolved.
PASS: W_UNRESOLVED_REFERENCE emitted.
PASS: unresolved required reference is classified TRACEABILITY_LOSS and is not mapping-eligible.


# Phase D

## Export hardening evidence

In [9]:
MEDICATION_SUMMARY.to_csv(
    OUTPUT_DIR
    / "medication_attribution_summary_v14.csv",
    index=False,
)

MEDICATION_REASON_COUNTS.to_csv(
    OUTPUT_DIR
    / "medication_attribution_reason_counts_v14.csv",
    index=False,
)

for variant, df in (
    MEDICATION_ATTRIBUTION.items()
):
    df.to_parquet(
        OUTPUT_DIR
        / f"{variant}_medication_attribution_audit_v14.parquet",
        index=False,
    )

REF_MICRO_AUDIT.to_csv(
    OUTPUT_DIR
    / "unresolved_reference_microtest_v14.csv",
    index=False,
)

fault_resource_path = (
    OUTPUT_DIR
    / "REF_MICRO_01_fault_resource.json"
)

fault_resource_path.write_text(
    json.dumps(
        fault_resource,
        indent=2,
    ),
    encoding="utf-8",
)

FINAL_GATE = pd.DataFrame([
    {
        "test_family": "medication_attribution_v4_response",
        "pass": bool(
            float(
                v4_row[
                    "attribution_warning_rate"
                ]
            )
            >
            v0_warning_rate
        ),
    },
    {
        "test_family": "unresolved_reference_detection",
        "pass": bool(
            baseline_resolves
            and not fault_resolves
            and REF_MICRO_AUDIT.iloc[0][
                "warning_code"
            ]
            == "W_UNRESOLVED_REFERENCE"
        ),
    },
    {
        "test_family": "unresolved_reference_lineage_consequence",
        "pass": bool(
            REF_MICRO_AUDIT.iloc[0][
                "fidelity_status"
            ]
            == "TRACEABILITY_LOSS"
            and not REF_MICRO_AUDIT.iloc[0][
                "mapping_eligible"
            ]
        ),
    },
])

display(
    FINAL_GATE
)

FINAL_GATE.to_csv(
    OUTPUT_DIR
    / "medication_reference_gate_v14.csv",
    index=False,
)

if not FINAL_GATE["pass"].all():
    raise RuntimeError(
        "P0.5/P0.6 hardening gate failed."
    )

print(
    "PASS: P0.5 medication-attribution hardening complete."
)

print(
    "PASS: P0.6 unresolved-reference micro-test complete."
)

print(
    "Outputs:",
    OUTPUT_DIR,
)

,test_family,pass
0,medication_attribution_v4_response,True
1,unresolved_reference_detection,True
2,unresolved_reference_lineage_consequence,True


PASS: P0.5 medication-attribution hardening complete.
PASS: P0.6 unresolved-reference micro-test complete.
Outputs: /content/drive/MyDrive/fhir_omop_colab/tfl_execution_v6/submission_freeze_v12/medication_reference_v14


# Interpretation boundary

The medication rule is a deterministic transformation-attribution check over the medication resource types actually present in the frozen project data. It is not clinical medication adjudication.

`REF_MICRO_01` is a focused fault-injection test, not a new V6 analytical cohort. It demonstrates that a reference known to resolve in frozen V0 becomes explicitly observable as `W_UNRESOLVED_REFERENCE` and `TRACEABILITY_LOSS` when deliberately broken.

After this notebook passes, the remaining P0 work is the one-shot submission-freeze runner and release/tag archive.